# Feature Extraction — RAVDESS

Run and inspect the pipeline feature extraction flow from `pipeline.extract_features`.

In [1]:
# Setup path and imports
from pathlib import Path
import sys

# Resolve SER_Project root from common notebook working directories
cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "SER_Project", cwd.parent, cwd.parent / "SER_Project"]
PROJECT_ROOT = next((p for p in candidates if (p / "pipeline" / "extract_features.py").exists()), None)

if PROJECT_ROOT is None:
    raise RuntimeError(f"Could not locate SER_Project/pipeline/extract_features.py from {cwd}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.extract_features import (
    build_feature_dataframe,
    discover_audio_files,
)

In [2]:
# Locate WAV files
# Guard against PROJECT_ROOT being None
if PROJECT_ROOT is None:
	possible = [cwd, cwd.parent, Path.cwd()]
	found = next((p for p in possible if (p / "data").exists()), None)
	if found is None:
		raise RuntimeError("Could not locate 'data' directory. Ensure PROJECT_ROOT is set or run the setup cell.")
	data_root = found / "data"
else:
	data_root = PROJECT_ROOT / "data"
audio_files = discover_audio_files(data_root)
print(f"Found {len(audio_files)} .wav files in {data_root}")
audio_files[:3]

Found 2452 .wav files in C:\Users\wblut\Documents\My Projects\Python\CSE 432\Project_CSE432-532\SER_Project\data


[WindowsPath('C:/Users/wblut/Documents/My Projects/Python/CSE 432/Project_CSE432-532/SER_Project/data/Actor_01/03-01-01-01-01-01-01.wav'),
 WindowsPath('C:/Users/wblut/Documents/My Projects/Python/CSE 432/Project_CSE432-532/SER_Project/data/Actor_01/03-01-01-01-01-02-01.wav'),
 WindowsPath('C:/Users/wblut/Documents/My Projects/Python/CSE 432/Project_CSE432-532/SER_Project/data/Actor_01/03-01-01-01-02-01-01.wav')]

In [3]:
# Extract summarized features into a DataFrame
features_df = build_feature_dataframe(
    filepaths=audio_files,
    sr=48000,
    n_mfcc=13,
    stats=["mean", "std"],
    include_song=False,
)

print("Shape:", features_df.shape)
features_df.head()

Extracting features:   0%|          | 0/2452 [00:00<?, ?file/s]c:\Users\wblut\Documents\My Projects\Python\CSE 432\Project_CSE432-532\SER_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Extracting features: 100%|██████████| 2452/2452 [09:27<00:00,  4.32file/s]


Shape: (1440, 384)


,filepath,filename,modality_code,vocal_channel_code,emotion_code,intensity_code,statement_code,repetition,actor_id,modality,...,rms_mean,rms_std,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std,duration_seconds,signal_abs_mean
0,C:\Users\wblut\Documents\My Projects\Python\CS...,03-01-01-01-01-01-01.wav,3,1,1,1,1,1,1,audio-only,...,0.002120,0.003391,7416.297748,4428.027505,5551.291828,1966.670942,13285.735887,7873.634242,3.303292,0.001663
1,C:\Users\wblut\Documents\My Projects\Python\CS...,03-01-01-01-01-01-02.wav,3,1,1,1,1,1,2,audio-only,...,0.003345,0.005519,4876.754825,3813.320523,4398.780650,2532.902086,9352.456012,7619.598807,3.636958,0.002492
2,C:\Users\wblut\Documents\My Projects\Python\CS...,03-01-01-01-01-01-03.wav,3,1,1,1,1,1,3,audio-only,...,0.003757,0.006070,4968.033485,3859.034171,4458.211223,2472.907411,9552.051084,7517.358922,3.436771,0.002893
3,C:\Users\wblut\Documents\My Projects\Python\CS...,03-01-01-01-01-01-04.wav,3,1,1,1,1,1,4,audio-only,...,0.002334,0.003605,5005.760327,4152.626602,4187.353901,2587.265324,9251.310484,7714.608188,3.303292,0.001800
4,C:\Users\wblut\Documents\My Projects\Python\CS...,03-01-01-01-01-01-05.wav,3,1,1,1,1,1,5,audio-only,...,0.001399,0.002330,6368.380142,3827.945641,5755.409712,2030.648976,12467.224482,7592.315579,3.603604,0.001039


In [4]:
# Quick sanity checks
display(features_df[["emotion", "vocal_channel"]].value_counts().head(12))

numeric_cols = features_df.select_dtypes(include=["number"]).columns
print("Numeric feature columns:", len(numeric_cols))
print("Missing numeric values:", int(features_df[numeric_cols].isna().sum().sum()))

emotion    vocal_channel
calm       speech           192
happy      speech           192
sad        speech           192
angry      speech           192
fearful    speech           192
disgust    speech           192
surprised  speech           192
neutral    speech            96
Name: count, dtype: int64

Numeric feature columns: 377
Missing numeric values: 0


In [5]:
# Save extracted features for modeling notebooks
output_csv = data_root / "features.csv"
features_df.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

Saved: C:\Users\wblut\Documents\My Projects\Python\CSE 432\Project_CSE432-532\SER_Project\data\features.csv
